# Final Exercise (Student) — Embedding Search and Recommendations with LightGCN

Use pretrained LightGCN embeddings to:
- Build user/item ID maps and a sparse interactions matrix from MovieLens data (no training).
- Load pretrained embeddings from disk and align them with your ID maps.
- Do item–item search by title with cosine similarity.
- Generate personalized Top-K recommendations per user.
- Optionally export predictions in the BLU12 evaluation format.

Notes:
- **You need to have the MovieLens 20M dataset extracted from the .zip file** in the `data/ml-20m/` directory.
- Run from the repo root so relative paths work: `./data/...`.
- If you don't have the pretrained `.pt` file locally, ask your instructor or set the correct path.
- Keep code simple and readable. Prefer vectorized ops over Python loops.


In [ ]:
# Imports and paths
import os
import json
import math
import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix

# Paths (adjust if needed)
ML20M_DIR = "./data/ml-10M100K"  # using MovieLens 10M ratings.dat format here
MOVIES_DAT = os.path.join(ML20M_DIR, "movies.dat")
RATINGS_DAT = os.path.join(ML20M_DIR, "ratings.dat")
# pretrained payload with E0/Z and id maps
EMB_PATH = "./data/lightgcn_ml20m.pt"

# Small helpers


def l2_normalize(mat: np.ndarray, axis: int = 1, eps: float = 1e-8) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=axis, keepdims=True)
    return mat / (norms + eps)


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Phase 1 — Build graph ID maps and CSR interactions (TODOs)

Goal: parse MovieLens data, build `user2idx`, `item2idx`, and a CSR matrix `R` with shape `[num_users, num_items]` where `R[u, i] = 1` if user u interacted with item i.

Data format hints:
- For `ml-10M100K/ratings.dat`, each line: `userId::movieId::rating::timestamp`.
- For `ml-10M100K/movies.dat`, each line: `movieId::Title (Year)::Genres`.

Tasks:
- Read `ratings.dat` efficiently with `pd.read_csv` (use `sep='::'`, `engine='python'`).
- Filter to implicit positives (rating >= 4.0 is a common rule-of-thumb).
- Build sorted unique lists of userIds and movieIds, then maps: `user2idx` and `item2idx`.
- Map the ratings to row/col indices and build a binary CSR matrix `R` of ones.
- Load `movies.dat` to build `itemId -> title` mapping (`item2title`).

Edge cases: ensure no NaNs; check that shapes look sensible.


In [ ]:
# TODO: Build user2idx, item2idx, item2title, and CSR R
# Steps:
# 1) Load ratings.dat (userId::movieId::rating::timestamp)
# 2) Filter to rating >= 3.5
# 3) Build sorted unique userIds, movieIds; create maps user2idx, item2idx
# 4) Map rows/cols and create CSR matrix R with data=1s
# 5) Load movies.dat and build item2title dict

# YOUR CODE HERE
raise NotImplementedError("Phase 1 TODO: build ID maps and CSR R")

## Phase 2 — Load pretrained LightGCN embeddings and align (TODOs)

A training script saved a payload at `EMB_PATH` (a `.pt` file) with keys like:
- `user2idx`, `item2idx`: the id maps used during training.
- `E0_user`, `E0_item`: initial embeddings.
- `Z_user`, `Z_item`: final propagated embeddings.
- `meta`: config and shapes.

Tasks:
- Load the payload with `torch.load(EMB_PATH, map_location='cpu')`.
- Align the item embeddings `Z_item` to your notebook's `item2idx`. Items missing from pretrained maps should be skipped.
- Produce two NumPy arrays: `U` and `I` of shapes `[num_users, d]` and `[num_items, d]` for user and item embeddings respectively. If you only have item embeddings, it's okay to set `U=None` for now.
- L2-normalize `I` row-wise for cosine similarity.

Tip: You may need to build an index mapping from your local `item2idx` to pretrained `item2idx`.


In [ ]:
# TODO: Load payload and align embeddings
# Steps:
# 1) payload = torch.load(EMB_PATH, map_location='cpu')
# 2) Note: look at the train.py 
# 2) Extract pretrained maps: pt_item2idx (and pt_user2idx if present)
# 3) Build an index map from local item2idx -> pretrained idx; allocate I with zeros
# 4) Fill I rows where items exist in pretrained; optionally do same for U if available
# 5) L2-normalize I across rows

# YOUR CODE HERE
raise NotImplementedError("Phase 2 TODO: load and align pretrained embeddings")

## Phase 3 — Item–item search by title (TODOs)

Goal: Given a query title substring, find the matching movie, then retrieve the top-N most similar items using cosine similarity in embedding space.

Tasks:
- Implement a simple lookup to find candidate itemIds by case-insensitive substring over titles; choose the best match.
- Compute cosine similarities: `sim = I @ I[q_idx]` assuming rows are L2-normalized.
- Exclude the query item itself; take top-N indices.
- Pretty-print results as a small table with title and score.

Edge cases: multiple matches; missing title; item not present in embeddings.


In [ ]:
# TODO: Implement item search and cosine similarity
# Inputs: item2title (dict), I (item embeddings, L2-normalized)
# Steps:
# - Choose a query string, e.g., q = "Matrix"
# - Find itemIds whose titles contain q (case-insensitive)
# - Pick one itemId -> get its local idx via item2idx
# - Compute sim = I @ I[q_idx]; take topN excluding q_idx
# - Display titles and scores

# YOUR CODE HERE
raise NotImplementedError("Phase 3 TODO: item–item search")

## Phase 4 — Personalized Top-K recommendations (TODOs)

Goal: For a given `userId`, recommend top-K items the user hasn't interacted with yet, using dot-product or cosine similarity between the user vector and item vectors.

Assumptions:
- If you have user embeddings `U`, use them directly. Otherwise, estimate a user vector by averaging their interacted item vectors from `I` (normalized mean is fine).
- Exclude items already in the user's history using `R`.

Tasks:
- Pick a userId with enough interactions.
- Build `u_vec` (from `U` or via history mean of `I`). Normalize it.
- Score all items: `scores = I @ u_vec`.
- Mask items in history; take top-K.
- Pretty-print titles and scores.


In [ ]:
# TODO: Implement personalized Top-K
# Inputs: R (csr), I (item embeddings), optional U (user embeddings), user2idx, item2idx
# Steps:
# - Choose a userId -> u = user2idx[userId]
# - If U is available: u_vec = U[u]; else compute mean of I over user history from R[u]
# - Normalize u_vec; compute scores = I @ u_vec
# - Mask items in user's history; take topK indices
# - Display titles and scores

# YOUR CODE HERE
raise NotImplementedError("Phase 4 TODO: personalized Top-K")

## Optional — Export predictions for BLU12 evaluation

If you want to evaluate recommendations with MAP@K using BLU12 tools:
- Create a CSV at `data/<pred_name>.csv` with no header.
- Column 0: `userId` as in the BLU12 ground truth split; columns 1..K: ranked `movieId`s.
- Then run from repo root:

python "01 - Essentials/BLU12 - Workflow/evaluation.py" <pred_name>

Keep user IDs consistent with the BLU12 test split.
